# Exploratory Data Analysis — MovieLens Latest Small

Complete EDA of the **MovieLens Latest Small** dataset using Pandas and Plotly.

**Dataset:** ~100k ratings from ~600 users on ~9k movies.

**Sections**
1. Dataset Overview
2. Missing Values Analysis
3. Ratings Analysis
4. User Analysis
5. Genre Analysis
6. Long Tail Analysis
7. Insights

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_DIR = Path("../data/raw/ml-latest-small/ml-latest-small")
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR.resolve()}"

PLOTLY_TEMPLATE = "plotly_white"
COLOR_PRIMARY = "#2E86AB"
COLOR_SECONDARY = "#E94F37"
COLOR_ACCENT = "#F6AE2D"

---
## 1. Dataset Overview

Load `movies.csv` and `ratings.csv`, inspect shapes, dtypes, and sample rows.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

print("=== movies.csv ===")
print(f"Shape: {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print(f"Columns: {list(movies.columns)}")
print(f"Dtypes:\n{movies.dtypes}\n")

print("=== ratings.csv ===")
print(f"Shape: {ratings.shape[0]:,} rows × {ratings.shape[1]} columns")
print(f"Columns: {list(ratings.columns)}")
print(f"Dtypes:\n{ratings.dtypes}\n")

print("=== Key counts ===")
print(f"Unique movies (catalog): {movies['movieId'].nunique():,}")
print(f"Unique movies (rated):   {ratings['movieId'].nunique():,}")
print(f"Unique users:            {ratings['userId'].nunique():,}")
print(f"Date range:              {ratings['timestamp'].min()} → {ratings['timestamp'].max()}")

In [ ]:
print("Sample — movies")
display(movies.head(10))

print("Sample — ratings")
display(ratings.head(10))

print("Descriptive statistics — ratings")
display(ratings["rating"].describe())

---
## 2. Missing Values Analysis

In [ ]:
def missing_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return missing-value summary for a DataFrame."""
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_missing": df.isna().sum().values,
        "pct_missing": (df.isna().mean() * 100).values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    })
    report.insert(0, "table", name)
    return report


missing = pd.concat(
    [missing_report(movies, "movies"), missing_report(ratings, "ratings")],
    ignore_index=True,
)
display(missing)

# Structural quality checks beyond nulls
empty_genres = (movies["genres"].fillna("").str.strip() == "").sum()
no_genre = (movies["genres"] == "(no genres listed)").sum()
dup_movies = movies["movieId"].duplicated().sum()
dup_ratings = ratings.duplicated(subset=["userId", "movieId"]).sum()
orphan_ratings = (~ratings["movieId"].isin(movies["movieId"])).sum()

print("Structural checks")
print(f"  Empty genre strings:           {empty_genres:,}")
print(f"  '(no genres listed)' movies:   {no_genre:,}")
print(f"  Duplicate movieId:             {dup_movies:,}")
print(f"  Duplicate (userId, movieId):   {dup_ratings:,}")
print(f"  Ratings with unknown movieId:  {orphan_ratings:,}")

In [ ]:
fig = px.bar(
    missing,
    x="column",
    y="pct_missing",
    color="table",
    barmode="group",
    title="Missing Values (%) by Column",
    labels={"pct_missing": "% Missing", "column": "Column", "table": "Table"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY, COLOR_SECONDARY],
)
fig.update_layout(yaxis_range=[0, max(5, missing["pct_missing"].max() * 1.2)])
fig.show()

---
## 3. Ratings Analysis

Distribution of ratings, average rating per movie, and most-rated titles.

In [ ]:
rating_counts = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_counts["pct"] = rating_counts["count"] / rating_counts["count"].sum() * 100

display(rating_counts)

fig = px.bar(
    rating_counts,
    x="rating",
    y="count",
    text=rating_counts["pct"].map(lambda x: f"{x:.1f}%"),
    title="Distribution of Ratings",
    labels={"rating": "Rating", "count": "Number of Ratings"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis=dict(dtick=0.5))
fig.show()

print(
    f"Mean={ratings['rating'].mean():.3f} | "
    f"Median={ratings['rating'].median():.3f} | "
    f"Std={ratings['rating'].std():.3f} | "
    f"Skew={ratings['rating'].skew():.3f}"
)

In [ ]:
movie_stats = (
    ratings.groupby("movieId", as_index=False)
    .agg(n_ratings=("rating", "size"), avg_rating=("rating", "mean"), std_rating=("rating", "std"))
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
)

fig = px.histogram(
    movie_stats,
    x="avg_rating",
    nbins=40,
    title="Distribution of Average Rating per Movie",
    labels={"avg_rating": "Average Rating", "count": "Number of Movies"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.add_vline(
    x=movie_stats["avg_rating"].mean(),
    line_dash="dash",
    line_color=COLOR_SECONDARY,
    annotation_text=f"mean={movie_stats['avg_rating'].mean():.2f}",
)
fig.show()

display(movie_stats[["avg_rating", "n_ratings"]].describe())

In [ ]:
TOP_N = 20

most_rated = movie_stats.nlargest(TOP_N, "n_ratings").sort_values("n_ratings")

fig = px.bar(
    most_rated,
    x="n_ratings",
    y="title",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="Blues",
    title=f"Top {TOP_N} Most Rated Movies",
    labels={"n_ratings": "Number of Ratings", "title": "Movie", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    hover_data={"avg_rating": ":.2f", "genres": True},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()

# Highly rated among movies with enough support
MIN_RATINGS = 50
top_quality = (
    movie_stats.query("n_ratings >= @MIN_RATINGS")
    .nlargest(TOP_N, "avg_rating")
    [["title", "avg_rating", "n_ratings", "genres"]]
)
print(f"Top {TOP_N} highest-rated movies (min {MIN_RATINGS} ratings)")
display(top_quality.reset_index(drop=True))

---
## 4. User Analysis

Activity patterns: ratings per user and most active users.

In [ ]:
user_stats = (
    ratings.groupby("userId", as_index=False)
    .agg(
        n_ratings=("rating", "size"),
        avg_rating=("rating", "mean"),
        std_rating=("rating", "std"),
        first_rating=("timestamp", "min"),
        last_rating=("timestamp", "max"),
    )
)

display(user_stats["n_ratings"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

fig = px.histogram(
    user_stats,
    x="n_ratings",
    nbins=50,
    title="Distribution of Ratings per User",
    labels={"n_ratings": "Ratings per User", "count": "Number of Users"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.add_vline(
    x=user_stats["n_ratings"].median(),
    line_dash="dash",
    line_color=COLOR_SECONDARY,
    annotation_text=f"median={user_stats['n_ratings'].median():.0f}",
)
fig.show()

In [ ]:
most_active = user_stats.nlargest(TOP_N, "n_ratings").sort_values("n_ratings")

fig = px.bar(
    most_active,
    x="n_ratings",
    y=most_active["userId"].astype(str),
    orientation="h",
    color="avg_rating",
    color_continuous_scale="OrRd",
    title=f"Top {TOP_N} Most Active Users",
    labels={"n_ratings": "Number of Ratings", "y": "User ID", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    hover_data={"avg_rating": ":.2f"},
)
fig.update_layout(yaxis_title="User ID", height=600)
fig.show()

fig = px.scatter(
    user_stats,
    x="n_ratings",
    y="avg_rating",
    opacity=0.5,
    title="User Activity vs Average Rating",
    labels={"n_ratings": "Ratings per User", "avg_rating": "Average Rating Given"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_traces(marker=dict(size=6))
fig.show()

---
## 5. Genre Analysis

Genre frequency in the catalog and average rating by genre (rating-weighted).

In [ ]:
movies_genres = movies.assign(genre=movies["genres"].str.split("|")).explode("genre")
movies_genres["genre"] = movies_genres["genre"].str.strip()

genre_freq = (
    movies_genres["genre"]
    .value_counts()
    .rename_axis("genre")
    .reset_index(name="n_movies")
)

fig = px.bar(
    genre_freq.sort_values("n_movies"),
    x="n_movies",
    y="genre",
    orientation="h",
    title="Genre Frequency in Movie Catalog",
    labels={"n_movies": "Number of Movies", "genre": "Genre"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_layout(height=650)
fig.show()

display(genre_freq)

In [ ]:
ratings_with_genres = ratings.merge(
    movies_genres[["movieId", "genre"]],
    on="movieId",
    how="inner",
)

genre_rating = (
    ratings_with_genres.groupby("genre", as_index=False)
    .agg(
        n_ratings=("rating", "size"),
        avg_rating=("rating", "mean"),
        n_movies=("movieId", "nunique"),
        n_users=("userId", "nunique"),
    )
    .sort_values("avg_rating", ascending=False)
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Average Rating by Genre", "Ratings Volume by Genre"),
    horizontal_spacing=0.18,
)

genre_by_avg = genre_rating.sort_values("avg_rating")
fig.add_trace(
    go.Bar(
        x=genre_by_avg["avg_rating"],
        y=genre_by_avg["genre"],
        orientation="h",
        marker_color=COLOR_PRIMARY,
        name="Avg Rating",
        hovertemplate="%{y}<br>avg=%{x:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
)

genre_by_vol = genre_rating.sort_values("n_ratings")
fig.add_trace(
    go.Bar(
        x=genre_by_vol["n_ratings"],
        y=genre_by_vol["genre"],
        orientation="h",
        marker_color=COLOR_ACCENT,
        name="N Ratings",
        hovertemplate="%{y}<br>n=%{x:,}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title_text="Genre Performance",
    template=PLOTLY_TEMPLATE,
    height=700,
    showlegend=False,
)
fig.update_xaxes(title_text="Average Rating", row=1, col=1, range=[genre_by_avg["avg_rating"].min() - 0.1, 5])
fig.update_xaxes(title_text="Number of Ratings", row=1, col=2)
fig.show()

display(genre_rating.reset_index(drop=True))

---
## 6. Long Tail Analysis

Popularity distribution: a small head of popular titles vs. a long tail of rarely rated movies.

In [ ]:
pop = movie_stats.sort_values("n_ratings", ascending=False).reset_index(drop=True)
pop["rank"] = np.arange(1, len(pop) + 1)
pop["cum_ratings"] = pop["n_ratings"].cumsum()
pop["cum_pct_ratings"] = pop["cum_ratings"] / pop["n_ratings"].sum() * 100
pop["cum_pct_movies"] = pop["rank"] / len(pop) * 100

# Coverage thresholds
for pct in [50, 80, 90]:
    n_needed = int((pop["cum_pct_ratings"] >= pct).idxmax()) + 1
    print(
        f"Top {n_needed:,} movies ({n_needed / len(pop) * 100:.1f}% of catalog) "
        f"account for {pct}% of all ratings"
    )

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Popularity Rank (log scale)", "Cumulative Rating Coverage"),
)

fig.add_trace(
    go.Scatter(
        x=pop["rank"],
        y=pop["n_ratings"],
        mode="lines",
        line=dict(color=COLOR_PRIMARY, width=2),
        name="Ratings",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=pop["cum_pct_movies"],
        y=pop["cum_pct_ratings"],
        mode="lines",
        line=dict(color=COLOR_SECONDARY, width=2),
        name="Coverage",
        fill="tozeroy",
        fillcolor="rgba(233, 79, 55, 0.15)",
    ),
    row=1,
    col=2,
)

fig.add_hline(y=80, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_xaxes(type="log", title_text="Movie Rank", row=1, col=1)
fig.update_yaxes(type="log", title_text="Number of Ratings", row=1, col=1)
fig.update_xaxes(title_text="% of Movies (by popularity)", row=1, col=2)
fig.update_yaxes(title_text="% of Ratings Covered", row=1, col=2)
fig.update_layout(title_text="Long-Tail Popularity Distribution", template=PLOTLY_TEMPLATE, height=450, showlegend=False)
fig.show()

In [ ]:
# Head vs tail buckets
thresholds = [1, 5, 10, 20, 50, 100]
bucket_rows = []
prev = 0
for t in thresholds:
    mask = (movie_stats["n_ratings"] > prev) & (movie_stats["n_ratings"] <= t)
    bucket_rows.append({
        "bucket": f"{prev + 1}–{t}",
        "n_movies": int(mask.sum()),
        "pct_movies": mask.mean() * 100,
        "n_ratings": int(movie_stats.loc[mask, "n_ratings"].sum()),
        "avg_rating": movie_stats.loc[mask, "avg_rating"].mean(),
    })
    prev = t

mask_head = movie_stats["n_ratings"] > thresholds[-1]
bucket_rows.append({
    "bucket": f">{thresholds[-1]}",
    "n_movies": int(mask_head.sum()),
    "pct_movies": mask_head.mean() * 100,
    "n_ratings": int(movie_stats.loc[mask_head, "n_ratings"].sum()),
    "avg_rating": movie_stats.loc[mask_head, "avg_rating"].mean(),
})

popularity_buckets = pd.DataFrame(bucket_rows)
popularity_buckets["pct_ratings"] = (
    popularity_buckets["n_ratings"] / popularity_buckets["n_ratings"].sum() * 100
)
display(popularity_buckets)

fig = px.bar(
    popularity_buckets,
    x="bucket",
    y=["pct_movies", "pct_ratings"],
    barmode="group",
    title="Share of Movies vs Share of Ratings by Popularity Bucket",
    labels={"value": "% Share", "bucket": "Ratings per Movie", "variable": "Metric"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY, COLOR_SECONDARY],
)
fig.show()

# Unrated catalog items
unrated = movies.loc[~movies["movieId"].isin(ratings["movieId"]), ["movieId", "title", "genres"]]
print(f"Movies in catalog with zero ratings: {len(unrated):,} ({len(unrated) / len(movies) * 100:.1f}%)")
display(unrated.head(10))

---
## 7. Insights Section

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
sparsity = 1 - (len(ratings) / (n_users * n_movies_rated))

top10_share = pop.head(10)["n_ratings"].sum() / pop["n_ratings"].sum() * 100
tail_1 = (movie_stats["n_ratings"] == 1).mean() * 100

insights = [
    {
        "area": "Scale",
        "finding": (
            f"Dataset has {len(ratings):,} ratings from {n_users:,} users on "
            f"{n_movies_rated:,} movies ({len(movies):,} in catalog)."
        ),
    },
    {
        "area": "Data quality",
        "finding": (
            "No missing values in core columns; genres use '|'-separated multi-labels; "
            f"{no_genre:,} movies marked '(no genres listed)'."
        ),
    },
    {
        "area": "Rating bias",
        "finding": (
            f"Ratings are left-skewed toward high scores "
            f"(mean={ratings['rating'].mean():.2f}, median={ratings['rating'].median():.2f}). "
            "Users tend to rate movies they like."
        ),
    },
    {
        "area": "Sparsity",
        "finding": (
            f"User–movie matrix sparsity is {sparsity * 100:.2f}% "
            f"({len(ratings):,} observed / {n_users * n_movies_rated:,} possible pairs)."
        ),
    },
    {
        "area": "User activity",
        "finding": (
            f"Median user rated {user_stats['n_ratings'].median():.0f} movies; "
            f"P95={user_stats['n_ratings'].quantile(0.95):.0f}. "
            "Activity is heavily skewed — a few power users dominate volume."
        ),
    },
    {
        "area": "Long tail",
        "finding": (
            f"Top 10 movies capture {top10_share:.1f}% of ratings; "
            f"{tail_1:.1f}% of rated movies have only 1 rating. "
            "Collaborative filtering will struggle on the cold tail."
        ),
    },
    {
        "area": "Genres",
        "finding": (
            f"Most common catalog genre: {genre_freq.iloc[0]['genre']} "
            f"({genre_freq.iloc[0]['n_movies']:,} movies). "
            f"Highest avg rating genre: {genre_rating.iloc[0]['genre']} "
            f"({genre_rating.iloc[0]['avg_rating']:.3f})."
        ),
    },
    {
        "area": "Modeling implications",
        "finding": (
            "Prefer hybrid recommenders (CF + content/genre features), "
            "apply popularity priors or minimum-support filters for ranking, "
            "and evaluate beyond accuracy (coverage, novelty, long-tail recall)."
        ),
    },
]

insights_df = pd.DataFrame(insights)
display(insights_df)

print("\n=== Summary KPIs ===")
kpi = pd.Series({
    "n_ratings": len(ratings),
    "n_users": n_users,
    "n_movies_catalog": len(movies),
    "n_movies_rated": n_movies_rated,
    "rating_mean": ratings["rating"].mean(),
    "rating_median": ratings["rating"].median(),
    "sparsity_pct": sparsity * 100,
    "ratings_per_user_median": user_stats["n_ratings"].median(),
    "ratings_per_movie_median": movie_stats["n_ratings"].median(),
})
display(kpi.to_frame("value"))

### Takeaways for downstream Graph RAG / recommender work

1. **Clean, dense head + sparse tail** — entity linking and retrieval will be reliable for popular titles; long-tail movies need content/metadata signals.
2. **Genre multi-label graph** — `|`-separated genres are a natural edge type (`MOVIE —[HAS_GENRE]→ GENRE`) for a knowledge graph.
3. **Positive-skew ratings** — treat absolute scores carefully; consider normalized / centered ratings per user.
4. **Temporal signal available** — `timestamp` enables time-aware splits (avoid random splits that leak future ratings).